# PainNAS on Google Colab: global 80/20 NAS and full LOSO

The default `global_search_loso` workflow performs one neural architecture search on a deterministic subject-disjoint 55/14/18 train/validation/architecture-selection-test split. Validation controls early stopping and pruning; mean subject-level macro-F1 on the test subjects ranks completed trials. It then evaluates the selected architecture with all 87 leave-one-subject-out (LOSO) folds. Because the test subjects participate in architecture selection, that later global LOSO summary remains exploratory. Set `WORKFLOW_MODE = 'cross_fitted'` to run the original five-block target-exclusive protocol instead.

> **Protocol warning:** the global workflow is exploratory. All 87 subject identities participate in the one-time architecture search, so the subsequent 87-fold result is not an unbiased nested-LOSO estimate. Each LOSO model nevertheless excludes its target from fitting and normalization, trains for the winning NAS trial's validation-best epoch count, and evaluates the target only afterward.

Select **Runtime → Change runtime type → GPU** before starting. Search databases, winning checkpoints, and completed LOSO folds are stored on Drive and can be resumed after a disconnect.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys
REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'painnas'
if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
REQUIRED_MODULES = ('painnas/search.py', 'painnas/loso.py', 'painnas/cross_fitted_loso.py')
assert all((PROJECT_DIR / path).is_file() for path in REQUIRED_MODULES)


## 2. Install pinned dependencies

In [ ]:
%pip -q install -U pip
%pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt
import optuna
import tensorflow as tf
print('Optuna:', optuna.__version__, 'TensorFlow:', tf.__version__)
assert tf.__version__.startswith('2.20.'), f'Unexpected TensorFlow version: {tf.__version__}'


## 3. Stage BioVid on the local Colab SSD

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a',
    drive_data_dir=DRIVE_DATA_DIR,
    local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)


## 4. Verify GPU and configure reproducibility

In [ ]:
import random
import numpy as np
import tensorflow as tf
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select a Colab GPU runtime.'
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)


## 5. Configure the NAS and LOSO workflow

`global_search_loso` runs one paper-scale 50-trial NAS followed by fresh training in every LOSO fold. `cross_fitted` preserves the five-block workflow with its 10-trial-per-block budget. Change `RUN_NAME` whenever any architecture, class, or training setting changes; saved manifests reject incompatible resumes.

In [ ]:
from painnas.config import PainNASConfig
WORKFLOW_MODE = 'global_search_loso'  # Alternatives: 'global_search_loso', 'cross_fitted'.
if WORKFLOW_MODE not in {'global_search_loso', 'cross_fitted'}:
    raise ValueError("WORKFLOW_MODE must be 'global_search_loso' or 'cross_fitted'.")
RUN_NAME = 'global_mc'
FUSION_MODE = 'early'  # Set to 'late' for the multi-model late-fusion NAS.
CLASS_INDICES = '0,1,2,3,4'  # Comma-separated raw BioVid labels, e.g. '0,1,2,3,4'.
RAW_CLASS_IDS = tuple(int(value.strip()) for value in CLASS_INDICES.split(',') if value.strip())
if len(RAW_CLASS_IDS) < 2 or len(RAW_CLASS_IDS) != len(set(RAW_CLASS_IDS)):
    raise ValueError('CLASS_INDICES must contain at least two unique comma-separated integers.')
EXPECTED_SUBJECTS = 87
SEARCH_VALIDATION_FRACTION = 0.20  # Applied to the 80% development subjects.
SEARCH_TEST_FRACTION = 0.20
SEARCH_VALIDATION_SUBJECTS = int(round(EXPECTED_SUBJECTS * SEARCH_VALIDATION_FRACTION))  # Nested-only count.
GLOBAL_N_TRIALS = 50
GLOBAL_SEARCH_MAX_EPOCHS = 50
CROSS_FITTED_N_TRIALS = 10
CROSS_FITTED_SEARCH_MAX_EPOCHS = 30
N_TRIALS = GLOBAL_N_TRIALS if WORKFLOW_MODE == 'global_search_loso' else CROSS_FITTED_N_TRIALS
SEARCH_MAX_EPOCHS = GLOBAL_SEARCH_MAX_EPOCHS if WORKFLOW_MODE == 'global_search_loso' else CROSS_FITTED_SEARCH_MAX_EPOCHS
OUTPUT_DIR = Path('/content/drive/MyDrive/PainNAS') / RUN_NAME
GLOBAL_DIR = OUTPUT_DIR / 'global_nas_loso'
SEARCH_DIR = GLOBAL_DIR / 'search'
LOSO_DIR = GLOBAL_DIR / 'loso'
CROSS_FITTED_DIR = OUTPUT_DIR / 'cross_fitted_loso'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = True
LOSO_START_INDEX = None  # One-based and inclusive.
LOSO_STOP_INDEX = None   # Set both values to run a resumable chunk.
MAX_FOLDS = None         # Debug-only cap after the index range.
VERBOSE = 1
CONFIG = PainNASConfig(
    seed=SEED,
    num_classes=len(RAW_CLASS_IDS),
    raw_class_ids=RAW_CLASS_IDS,
    batch_size=128,
    n_trials=N_TRIALS,
    search_max_epochs=SEARCH_MAX_EPOCHS,
    loso_max_epochs=100,
    cross_fitted_continuation_epochs=None,  # Maximum epochs; source-only early stopping may finish sooner.
    search_patience=8,
    loso_patience=15,
    search_validation_subjects=SEARCH_VALIDATION_SUBJECTS,
    outer_block_count=5,
    inner_fold_count=3,
    uncertainty_beta=1.0,
    max_parameters=40_000_000,
    bootstrap_samples=10_000,
    fusion_mode=FUSION_MODE,
)
assert CONFIG.expected_subjects == EXPECTED_SUBJECTS
print(CONFIG)
print('Workflow:', WORKFLOW_MODE)
print('NAS budget:', CONFIG.n_trials, 'trials x', CONFIG.search_max_epochs, 'maximum epochs')
print('Output:', GLOBAL_DIR if WORKFLOW_MODE == 'global_search_loso' else CROSS_FITTED_DIR)


## 6. Load BioVid and audit subject isolation

In [ ]:
import pandas as pd
from IPython.display import display
from painnas.data import (
    build_cross_fitted_subject_plan,
    build_loso_fold_indices,
    deterministic_global_search_subject_split,
    indices_for_subjects,
    load_biovid_binary,
)
ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
ALL_SUBJECTS = tuple(int(value) for value in sorted(ARRAYS.unique_subjects.tolist()))
assert len(ALL_SUBJECTS) == EXPECTED_SUBJECTS

loso_audit_rows = []
for fold_index, target_subject in enumerate(ALL_SUBJECTS, start=1):
    fold = build_loso_fold_indices(ARRAYS, target_subject)
    train_subjects = set(map(int, ARRAYS.subjects[fold.train]))
    validation_subjects = set(map(int, ARRAYS.subjects[fold.validation]))
    test_subjects = set(map(int, ARRAYS.subjects[fold.test]))
    assert target_subject not in train_subjects
    assert target_subject not in validation_subjects
    assert test_subjects == {target_subject}
    assert len(fold.source_subjects) == EXPECTED_SUBJECTS - 1
    loso_audit_rows.append({
        'fold': fold_index,
        'target_subject': target_subject,
        'source_subjects': len(fold.source_subjects),
        'train_samples': len(fold.train),
        'unused_source_test_samples': len(fold.validation),
        'target_test_samples': len(fold.test),
    })
print('Audited target-exclusive LOSO folds:', len(loso_audit_rows))
display(pd.DataFrame(loso_audit_rows).head())

if WORKFLOW_MODE == 'global_search_loso':
    SEARCH_SUBJECT_SPLIT = deterministic_global_search_subject_split(
        ALL_SUBJECTS, seed=CONFIG.seed,
        test_fraction=SEARCH_TEST_FRACTION,
        validation_fraction=SEARCH_VALIDATION_FRACTION,
    )
    SEARCH_TRAIN_SUBJECTS = SEARCH_SUBJECT_SPLIT.train_subjects
    SEARCH_VALIDATION_SUBJECTS = SEARCH_SUBJECT_SPLIT.validation_subjects
    SEARCH_TEST_SUBJECTS = SEARCH_SUBJECT_SPLIT.test_subjects
    train_set = set(SEARCH_TRAIN_SUBJECTS)
    validation_set = set(SEARCH_VALIDATION_SUBJECTS)
    test_set = set(SEARCH_TEST_SUBJECTS)
    assert len(SEARCH_TRAIN_SUBJECTS) == 55
    assert len(SEARCH_VALIDATION_SUBJECTS) == 14
    assert len(SEARCH_TEST_SUBJECTS) == 18
    assert train_set.isdisjoint(validation_set)
    assert train_set.isdisjoint(test_set)
    assert validation_set.isdisjoint(test_set)
    assert train_set | validation_set | test_set == set(ALL_SUBJECTS)
    search_train_indices = indices_for_subjects(
        ARRAYS, SEARCH_TRAIN_SUBJECTS, split_code=ARRAYS.train_split_code
    )
    search_validation_indices = indices_for_subjects(
        ARRAYS, SEARCH_VALIDATION_SUBJECTS, split_code=ARRAYS.test_split_code
    )
    search_test_indices = indices_for_subjects(
        ARRAYS, SEARCH_TEST_SUBJECTS, split_code=ARRAYS.test_split_code
    )
    assert len(search_train_indices) and len(search_validation_indices) and len(search_test_indices)
    split_audit = pd.DataFrame([
        {
            'split': 'NAS train',
            'subject_count': len(SEARCH_TRAIN_SUBJECTS),
            'subject_fraction': len(SEARCH_TRAIN_SUBJECTS) / len(ALL_SUBJECTS),
            'sample_count': len(search_train_indices),
            'predefined_samples': 'Train',
            'subjects': SEARCH_TRAIN_SUBJECTS,
        },
        {
            'split': 'NAS validation',
            'subject_count': len(SEARCH_VALIDATION_SUBJECTS),
            'subject_fraction': len(SEARCH_VALIDATION_SUBJECTS) / len(ALL_SUBJECTS),
            'sample_count': len(search_validation_indices),
            'predefined_samples': 'Test',
            'subjects': SEARCH_VALIDATION_SUBJECTS,
        },
        {
            'split': 'NAS architecture-selection test',
            'subject_count': len(SEARCH_TEST_SUBJECTS),
            'subject_fraction': len(SEARCH_TEST_SUBJECTS) / len(ALL_SUBJECTS),
            'sample_count': len(search_test_indices),
            'predefined_samples': 'Test',
            'subjects': SEARCH_TEST_SUBJECTS,
        },
    ])
    display(split_audit)
    print('WARNING: global architecture selection makes the final LOSO summary exploratory.')
else:
    SUBJECT_PLAN = build_cross_fitted_subject_plan(
        ALL_SUBJECTS,
        outer_block_count=CONFIG.outer_block_count,
        inner_fold_count=CONFIG.inner_fold_count,
        seed=CONFIG.seed,
    )
    plan_rows = []
    for block_index, (block, inner_folds) in enumerate(
        zip(SUBJECT_PLAN.outer_blocks, SUBJECT_PLAN.inner_folds_by_block), start=1
    ):
        plan_rows.append({
            'outer_block': block_index,
            'outer_subject_count': len(block),
            'development_subject_count': sum(map(len, inner_folds)),
            'inner_fold_sizes': tuple(map(len, inner_folds)),
            'outer_subjects': block,
        })
    assert sorted(s for block in SUBJECT_PLAN.outer_blocks for s in block) == list(ALL_SUBJECTS)
    display(pd.DataFrame(plan_rows))
print('Samples:', len(ARRAYS.y), 'Shape:', ARRAYS.X.shape)


## 7. Inspect the fixed trial-0 baseline

In [ ]:
from painnas.model import ArchitectureSpec, LateFusionArchitectureSpec, build_model
BASELINE = LateFusionArchitectureSpec.baseline() if CONFIG.fusion_mode == 'late' else ArchitectureSpec.baseline()
BASELINE_MODEL = build_model(
    BASELINE,
    input_shape=(ARRAYS.num_modalities, ARRAYS.sequence_length, 1),
    num_classes=CONFIG.num_classes,
    modalities=CONFIG.modalities,
)
print(BASELINE)
print('Parameters:', f'{BASELINE_MODEL.count_params():,}')
del BASELINE_MODEL


## 8. Run or resume the selected workflow

In global mode, this first completes the deterministic 55/14/18 Optuna study. Training subjects fit each model, validation subjects control early stopping and pruning, and the mean subject-level macro-F1 on the 18 architecture-selection test subjects ranks completed trials. The winner is then trained from scratch in every selected LOSO fold for exactly its validation-best epoch count. Source-subject `Test` samples are not used for LOSO early stopping. Architecture, hyperparameters, and epoch count transfer from NAS; weights and optimizer state never transfer. Because the test subjects select the architecture, the later global LOSO summary remains exploratory. In cross-fitted mode, the original five target-exclusive block searches and warm-started continuations are retained. Rerun this cell after a disconnect to resume.

In [ ]:
from painnas.cross_fitted_loso import run_cross_fitted_loso_nas
from painnas.loso import run_loso
from painnas.search import load_architecture, run_search

if WORKFLOW_MODE == 'global_search_loso':
    SEARCH_RESULT = run_search(
        ARRAYS,
        CONFIG,
        SEARCH_DIR,
        resume=RESUME,
        verbose=VERBOSE,
        train_subjects=SEARCH_TRAIN_SUBJECTS,
        validation_subjects=SEARCH_VALIDATION_SUBJECTS,
        test_subjects=SEARCH_TEST_SUBJECTS,
        study_name=f'painnas_{CONFIG.fusion_mode}_global_55_14_18',
    )
    SELECTED_ARCHITECTURE = load_architecture(Path(SEARCH_RESULT['best_architecture_path']))
    SELECTED_MODEL = build_model(
        SELECTED_ARCHITECTURE,
        input_shape=(ARRAYS.num_modalities, ARRAYS.sequence_length, 1),
        num_classes=CONFIG.num_classes,
        modalities=CONFIG.modalities,
    )
    assert SELECTED_MODEL.count_params() <= CONFIG.max_parameters
    display(pd.DataFrame([{
        'best_trial': SEARCH_RESULT['best_trial_number'],
        'validation_macro_f1': SEARCH_RESULT['best_validation_macro_f1'],
        'test_subject_macro_f1_mean': SEARCH_RESULT['best_test_subject_macro_f1_mean'],
        'test_subject_macro_f1_std': SEARCH_RESULT['best_test_subject_macro_f1_std'],
        'best_epoch': SEARCH_RESULT['best_epoch'],
        'parameter_count': SELECTED_MODEL.count_params(),
        'architecture_path': SEARCH_RESULT['best_architecture_path'],
    }]))
    print(SELECTED_ARCHITECTURE)
    del SELECTED_MODEL
    tf.keras.backend.clear_session()
    SUMMARY = run_loso(
        ARRAYS,
        SELECTED_ARCHITECTURE,
        CONFIG,
        LOSO_DIR,
        resume=RESUME,
        training_epochs=int(SEARCH_RESULT['best_epoch']),
        start_index=LOSO_START_INDEX,
        stop_index=LOSO_STOP_INDEX,
        max_folds=MAX_FOLDS,
        verbose=VERBOSE,
    )
    if LOSO_START_INDEX is None and LOSO_STOP_INDEX is None and MAX_FOLDS is None:
        assert SUMMARY['completed_folds'] == len(ALL_SUBJECTS)
else:
    SUMMARY = run_cross_fitted_loso_nas(
        ARRAYS,
        CONFIG,
        CROSS_FITTED_DIR,
        resume=RESUME,
        start_index=LOSO_START_INDEX,
        stop_index=LOSO_STOP_INDEX,
        max_folds=MAX_FOLDS,
        verbose=VERBOSE,
    )
display(pd.DataFrame([SUMMARY['metrics']['accuracy'], SUMMARY['metrics']['macro_f1']], index=['accuracy', 'macro_f1']))


## 9. Optional runtime cleanup

In [ ]:
# Release model resources, then disconnect and delete the hosted Colab runtime.
import gc
import logging
import sys

try:
    tf.keras.backend.clear_session()
except NameError:
    pass
gc.collect()
logging.shutdown()

try:
    from google.colab import runtime
except ImportError:
    print("Cleanup complete; no hosted Colab runtime was detected.")
else:
    print("Cleanup complete; disconnecting and deleting the Colab runtime.")
    sys.stdout.flush()
    sys.stderr.flush()
    runtime.unassign()